# Diagnóstico

Este notebook revisa la calidad de las fuentes recibidas.

## Qué cubre

- Perfilamiento por fuente: volumen, completitud, distribución y consistencia.
- Hallazgos priorizados con evidencia reproducible en SQL.
- Reglas de calidad para que los problemas no pasen sin ser detectados.
- Preguntas al dueño de negocio para cerrar ambigüedades antes de calcular el ausentismo oficial.

## Librerías

In [55]:
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import Markdown, display

## Parámetros

In [56]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

In [57]:
# Directorio
ROOT = Path.cwd()

if not (ROOT / 'data' / 'landing').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data' / 'landing'
SQL_DIR = ROOT / 'sql' / 'diagnostico'

print(f'Raíz de trabajo: {ROOT}')
print(f'Directorio de datos: {DATA_DIR}')
print(f'Directorio SQL: {SQL_DIR}')

Raíz de trabajo: /home/ramalambom/Documents/Don Doctor/code
Directorio de datos: /home/ramalambom/Documents/Don Doctor/code/data/landing
Directorio SQL: /home/ramalambom/Documents/Don Doctor/code/sql/diagnostico


In [58]:
north_path = (DATA_DIR / 'ips_norte_citas.csv').as_posix()
sur_path = (DATA_DIR / 'ips_sur_citas.csv').as_posix()
occ_path = (DATA_DIR / 'ips_occidente_citas.csv').as_posix()

In [59]:
conn = duckdb.connect()
CORTE_EXTRACCION = pd.Timestamp('2026-06-30 23:59:00')

## Datos

In [60]:
def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

In [61]:
# IPS Norte
conn.execute(
    f"""
    CREATE OR REPLACE VIEW citas_norte_raw AS
    SELECT *
    FROM read_csv_auto({sql_literal(north_path)})
    """
)

# IPS Sur
conn.execute(
    f"""
    CREATE OR REPLACE VIEW citas_sur_raw AS
    SELECT *
    FROM read_csv_auto({sql_literal(sur_path)})
    """
)

# IPS Occidente
conn.execute(
    f"""
    CREATE OR REPLACE VIEW citas_occidente_raw AS
    SELECT *
    FROM read_csv_auto({sql_literal(occ_path)})
    """
)

wa_raw = pd.read_json(DATA_DIR / 'whatsapp_eventos.jsonl', lines=True)


## Exploración inicial

In [62]:
source_shapes_csv = conn.execute("""
WITH shapes AS (
    SELECT
        'NORTE' AS fuente,
        COUNT(*) AS filas,
        (
            SELECT COUNT(*)
            FROM (
                DESCRIBE SELECT * FROM citas_norte_raw
            )
        ) AS columnas
    FROM
        citas_norte_raw

    UNION ALL

    SELECT
        'SUR' AS fuente,
        COUNT(*) AS filas,
        (
            SELECT COUNT(*)
            FROM (
                DESCRIBE SELECT * FROM citas_sur_raw
            )
        ) AS columnas
    FROM
        citas_sur_raw

    UNION ALL

    SELECT
        'OCCIDENTE' AS fuente,
        COUNT(*) AS filas,
        (
            SELECT COUNT(*)
            FROM (
                DESCRIBE SELECT * FROM citas_occidente_raw
            )
        ) AS columnas
    FROM
        citas_occidente_raw
)

SELECT *
FROM shapes
ORDER BY fuente
""").fetchdf()

source_shapes = pd.concat(
    [
        source_shapes_csv,
        pd.DataFrame([
            {
                'fuente': 'WHATSAPP',
                'filas': len(wa_raw),
                'columnas': len(wa_raw.columns),
            }
        ]),
    ],
    ignore_index=True,
)

display(source_shapes)

,fuente,filas,columnas
0,NORTE,6555,20
1,OCCIDENTE,4228,20
2,SUR,5312,22
3,WHATSAPP,38416,5


In [63]:
conn.execute("""
CREATE OR REPLACE VIEW citas_norte AS
    SELECT
        'NORTE' AS fuente,
        cita_id,
        paciente_id,
        CAST(NULL AS VARCHAR) AS documento_identidad,
        CAST(NULL AS VARCHAR) AS telefono,
        edad,
        sexo AS sexo_crudo,
        sexo AS sexo_homologado,
        regimen,
        localidad,
        especialidad,
        medico_id,
        sede,
        canal_agendamiento,
        CAST(fecha_creacion AS TIMESTAMP) AS fecha_creacion,
        CAST(fecha_cita AS TIMESTAMP) AS fecha_cita,
        CAST(fecha_actualizacion AS TIMESTAMP) AS fecha_actualizacion,
        estado AS estado_crudo,
        estado AS estado_homologado,
        recordatorio_enviado,
        confirmada,
        gestion_recuperacion,
        motivo_cierre,
        observaciones,
        cita_origen_id

    FROM
        citas_norte_raw
""")

conn.execute("""
CREATE OR REPLACE VIEW citas_sur AS
    SELECT
        'SUR' AS fuente,
        cita_id,
        paciente_id,
        CAST(documento_identidad AS VARCHAR) AS documento_identidad,
        CAST(telefono AS VARCHAR) AS telefono,
        edad,
        sexo AS sexo_crudo,
        sexo AS sexo_homologado,
        regimen,
        localidad,
        especialidad,
        medico_id,
        sede,
        canal_agendamiento,
        CAST(fecha_creacion AS TIMESTAMP) AS fecha_creacion,
        CAST(fecha_cita AS TIMESTAMP) AS fecha_cita,
        CAST(fecha_actualizacion AS TIMESTAMP) AS fecha_actualizacion,
        estado AS estado_crudo,
        estado AS estado_homologado,
        recordatorio_enviado,
        confirmada,
        gestion_recuperacion,
        motivo_cierre,
        observaciones,
        cita_origen_id

    FROM
        citas_sur_raw
""")

conn.execute("""
CREATE OR REPLACE VIEW citas_occidente AS
    SELECT
        'OCCIDENTE' AS fuente,
        id_cita AS cita_id,
        id_paciente AS paciente_id,
        CAST(NULL AS VARCHAR) AS documento_identidad,
        CAST(NULL AS VARCHAR) AS telefono,
        edad_paciente AS edad,
        genero AS sexo_crudo,
        CASE
            WHEN genero = 'Femenino' THEN 'F'
            WHEN genero = 'Masculino' THEN 'M'
            ELSE genero
        END AS sexo_homologado,
        regimen_salud AS regimen,
        localidad,
        especialidad,
        medico_id,
        sede,
        canal_agendamiento,
        CAST(fecha_creacion AS TIMESTAMP) AS fecha_creacion,
        CAST(fecha_hora_cita AS TIMESTAMP) AS fecha_cita,
        CAST(fecha_actualizacion AS TIMESTAMP) AS fecha_actualizacion,
        estado_cita AS estado_crudo,
        CASE
            WHEN estado_cita = 'ATD' THEN 'ATENDIDA'
            WHEN estado_cita = 'CAN' THEN 'CANCELADA'
            WHEN estado_cita = 'CONF' THEN 'CONFIRMADA'
            WHEN estado_cita = 'NAS' THEN 'NO_ASISTIO'
            WHEN estado_cita = 'PEN' THEN 'PENDIENTE'
            WHEN estado_cita = 'REP' THEN 'REAGENDADA'
            ELSE estado_cita
        END AS estado_homologado,
        recordatorio_enviado,
        confirmada,
        gestion_recuperacion,
        motivo_cierre,
        observaciones,
        id_cita_origen AS cita_origen_id

    FROM
        citas_occidente_raw
""")

conn.execute("""
CREATE OR REPLACE VIEW citas_std AS
    SELECT * FROM citas_norte

    UNION ALL

    SELECT * FROM citas_sur

    UNION ALL

    SELECT * FROM citas_occidente
""")

In [64]:
wa = wa_raw.copy()
wa['ips'] = wa['contexto'].apply(
    lambda value: value.get('ips') if isinstance(value, dict) else None
)
wa['ref_cita'] = wa['contexto'].apply(
    lambda value: value.get('ref_cita') if isinstance(value, dict) else None
)
wa['destinatario'] = wa['mensaje'].apply(
    lambda value: value.get('destinatario') if isinstance(value, dict) else None
)
wa['plantilla'] = wa['mensaje'].apply(
    lambda value: value.get('plantilla') if isinstance(value, dict) else None
)
wa['cuerpo'] = wa['mensaje'].apply(
    lambda value: value.get('cuerpo') if isinstance(value, dict) else None
)
wa['fecha_evento'] = pd.to_datetime(wa['ts'], unit='ms', errors='coerce')

citas_std_preview = conn.execute('SELECT * FROM citas_std LIMIT 5').fetchdf()
citas_std_preview

,fuente,cita_id,paciente_id,documento_identidad,telefono,edad,sexo_crudo,sexo_homologado,regimen,localidad,especialidad,medico_id,sede,canal_agendamiento,fecha_creacion,fecha_cita,fecha_actualizacion,estado_crudo,estado_homologado,recordatorio_enviado,confirmada,gestion_recuperacion,motivo_cierre,observaciones,cita_origen_id
0,NORTE,NOR-185255,8039e795eec7,None,None,28,M,M,Contributivo,San Cristóbal,Medicina general,MED-N008,Sede Norte 3,voz,2026-03-10 04:40:00,2026-03-14 13:40:00,2026-03-15 15:40:00,ATENDIDA,ATENDIDA,0,0,0,NaN,None,NaN
1,NORTE,NOR-183392,52f9a4264520,None,None,48,F,F,Contributivo,Engativá,Dermatología,MED-N003,Sede Norte 1,call_center,2026-02-22 02:00:00,2026-03-05 09:00:00,2026-03-05 16:00:00,NO_ASISTIO,NO_ASISTIO,0,0,1,Paciente no se presentó,None,NaN
2,NORTE,NOR-188355,f12c96152f44,None,None,41,F,F,Particular,Fontibón,Cardiología,MED-N037,Sede Norte 1,call_center,2025-07-20 08:20:00,2025-08-01 08:20:00,2025-08-02 01:20:00,ATENDIDA,ATENDIDA,1,1,0,NaN,None,NOR-186418
3,NORTE,NOR-185046,5c2a54573057,None,None,30,F,F,Particular,Barrios Unidos,Odontología,MED-N036,Sede Norte 2,voz,2026-06-04 06:20:00,2026-06-23 14:20:00,2026-06-23 20:20:00,ATENDIDA,ATENDIDA,1,0,0,NaN,None,NaN
4,NORTE,NOR-186004,840f27a3f6fb,None,None,12,F,F,Subsidiado,Chapinero,Nutrición,MED-N004,Sede Norte 2,web,2026-05-26 04:00:00,2026-05-27 08:00:00,2026-05-27 18:00:00,NO_ASISTIO,NO_ASISTIO,1,0,1,NaN,None,NaN


## 1. Perfilamiento por fuente

En esta parte se revisa cómo llega cada fuente. La idea es responder cuatro preguntas simples:

- Cuánto dato llegó.
- Qué tan completo llega.
- Cómo se reparten los valores más importantes.
- Qué tan consistente es la información dentro de cada fuente y entre fuentes.

In [65]:
perfil_citas = conn.execute(
    """
    WITH columnas AS (
        SELECT
            'NORTE' AS fuente,
            COUNT(*) AS columnas
        FROM (
            DESCRIBE SELECT * FROM citas_norte_raw
        )

        UNION ALL

        SELECT
            'SUR' AS fuente,
            COUNT(*) AS columnas
        FROM (
            DESCRIBE SELECT * FROM citas_sur_raw
        )

        UNION ALL

        SELECT
            'OCCIDENTE' AS fuente,
            COUNT(*) AS columnas
        FROM (
            DESCRIBE SELECT * FROM citas_occidente_raw
        )
    ),
    duplicados AS (
        SELECT
            fuente,
            SUM(filas_duplicadas) AS duplicados_por_cita_id
        FROM (
            SELECT
                fuente,
                cita_id,
                COUNT(*) - 1 AS filas_duplicadas
            FROM
                citas_std
            GROUP BY
                1, 2
            HAVING
                COUNT(*) > 1
        )
        GROUP BY 1
    ),
    base AS (
        SELECT
            fuente,
            COUNT(*) AS filas,
            COUNT(DISTINCT cita_id) AS citas_unicas,
            COUNT(DISTINCT paciente_id) AS pacientes_unicos,
            MIN(fecha_creacion) AS min_fecha_creacion,
            MAX(fecha_creacion) AS max_fecha_creacion,
            MIN(fecha_cita) AS min_fecha_cita,
            MAX(fecha_cita) AS max_fecha_cita
        FROM
            citas_std
        GROUP BY
            1
    )

    SELECT
        base.fuente,
        base.filas,
        columnas.columnas,
        base.citas_unicas,
        base.pacientes_unicos,
        COALESCE(duplicados.duplicados_por_cita_id, 0) AS duplicados_por_cita_id,
        base.min_fecha_creacion,
        base.max_fecha_creacion,
        base.min_fecha_cita,
        base.max_fecha_cita
    FROM
        base
    LEFT JOIN
        columnas USING (fuente)
    LEFT JOIN
        duplicados USING (fuente)
    ORDER BY
        base.fuente
""").fetchdf()

perfil_whatsapp = pd.DataFrame([
    {
        'fuente': 'WHATSAPP',
        'filas': len(wa),
        'columnas': len(wa_raw.columns),
        'eventos_unicos': wa['_id'].nunique(),
        'eventos_con_ref_cita': wa['ref_cita'].notna().sum(),
        'eventos_sin_ref_cita': wa['ref_cita'].isna().sum(),
        'destinatarios_unicos': wa['destinatario'].nunique(),
        'min_fecha_evento': wa['fecha_evento'].min(),
        'max_fecha_evento': wa['fecha_evento'].max(),
    }
])

display(Markdown('### Tabla de perfilamiento por fuente'))
display(perfil_citas)
display(perfil_whatsapp)

### Tabla de perfilamiento por fuente

,fuente,filas,columnas,citas_unicas,pacientes_unicos,duplicados_por_cita_id,min_fecha_creacion,max_fecha_creacion,min_fecha_cita,max_fecha_cita
0,NORTE,6555,20,6357,2255,198.0,2025-05-30 03:00:00,2026-08-02 12:00:00,2025-07-01 08:20:00,2026-08-13 13:20:00
1,OCCIDENTE,4228,20,4228,1513,0.0,2025-06-01 06:40:00,2026-08-04 13:00:00,2025-07-01 14:00:00,2026-08-12 21:40:00
2,SUR,5312,22,5312,1885,0.0,2025-05-26 00:00:00,2026-08-02 09:40:00,2025-07-01 08:00:00,2026-08-12 14:20:00


,fuente,filas,columnas,eventos_unicos,eventos_con_ref_cita,eventos_sin_ref_cita,destinatarios_unicos,min_fecha_evento,max_fecha_evento
0,WHATSAPP,38416,5,36621,27065,11351,5822,2025-06-29 15:20:00,2026-08-13 00:41:00


In [66]:
campos_citas = [
    'cita_id', 'paciente_id', 'documento_identidad', 'telefono', 'edad', 'sexo_crudo',
    'regimen', 'canal_agendamiento', 'fecha_creacion', 'fecha_cita', 'fecha_actualizacion',
    'estado_crudo', 'recordatorio_enviado', 'confirmada', 'gestion_recuperacion',
    'motivo_cierre', 'observaciones', 'cita_origen_id'
]

completitud_citas_sql = '\nUNION ALL\n'.join([
    f"""
    SELECT
        fuente,
        '{campo}' AS campo,
        ROUND(
            100.0 * AVG(CASE WHEN {campo} IS NOT NULL THEN 1 ELSE 0 END),
            2
        ) AS completitud_pct
    FROM
        citas_std
    GROUP BY
        fuente
    """.strip()
    for campo in campos_citas
])
completitud_citas = conn.execute(completitud_citas_sql).fetchdf()

completitud_whatsapp = pd.DataFrame([
    {'fuente': 'WHATSAPP', 'campo': '_id', 'completitud_pct': round((1 - wa['_id'].isna().mean()) * 100, 2)},
    {'fuente': 'WHATSAPP', 'campo': 'tipo', 'completitud_pct': round((1 - wa['tipo'].isna().mean()) * 100, 2)},
    {'fuente': 'WHATSAPP', 'campo': 'ips', 'completitud_pct': round((1 - wa['ips'].isna().mean()) * 100, 2)},
    {'fuente': 'WHATSAPP', 'campo': 'ref_cita', 'completitud_pct': round((1 - wa['ref_cita'].isna().mean()) * 100, 2)},
    {'fuente': 'WHATSAPP', 'campo': 'destinatario', 'completitud_pct': round((1 - wa['destinatario'].isna().mean()) * 100, 2)},
    {'fuente': 'WHATSAPP', 'campo': 'plantilla', 'completitud_pct': round((1 - wa['plantilla'].isna().mean()) * 100, 2)},
    {'fuente': 'WHATSAPP', 'campo': 'fecha_evento', 'completitud_pct': round((1 - wa['fecha_evento'].isna().mean()) * 100, 2)},
])


In [67]:
campos_criticos = ['cita_id', 'paciente_id', 'documento_identidad', 'telefono', 'fecha_cita', 'estado_crudo', 'motivo_cierre', 'observaciones', 'cita_origen_id']
display(Markdown('### Completitud de campos clave en citas'))
display(
    completitud_citas[completitud_citas['campo'].isin(campos_criticos)].sort_values('completitud_pct')
)

display(Markdown('### Completitud de campos clave en WhatsApp'))
display(completitud_whatsapp)

### Completitud de campos clave en citas

,fuente,campo,completitud_pct
6,NORTE,documento_identidad,0.00
11,OCCIDENTE,telefono,0.00
10,NORTE,telefono,0.00
8,OCCIDENTE,documento_identidad,0.00
50,OCCIDENTE,observaciones,4.49
48,SUR,observaciones,4.71
49,NORTE,observaciones,4.84
52,OCCIDENTE,cita_origen_id,5.39
51,NORTE,cita_origen_id,5.52
53,SUR,cita_origen_id,5.87


### Completitud de campos clave en WhatsApp

,fuente,campo,completitud_pct
0,WHATSAPP,_id,100.00
1,WHATSAPP,tipo,100.00
2,WHATSAPP,ips,100.00
3,WHATSAPP,ref_cita,70.45
4,WHATSAPP,destinatario,100.00
5,WHATSAPP,plantilla,100.00
6,WHATSAPP,fecha_evento,100.00


In [68]:
conteo_unicos_sql = "\nUNION ALL\n".join([
    f"""
    SELECT
        fuente,
        '{campo}' AS campo,
        COUNT(DISTINCT {campo}) AS valores_unicos
    FROM
        citas_std
    GROUP BY
        fuente
    """.strip()
    for campo in campos_citas
])

conteo_unicos_por_columna = conn.execute(conteo_unicos_sql).fetchdf()
tabla_unicos = (
    conteo_unicos_por_columna
    .pivot(index='fuente', columns='campo', values='valores_unicos')
    .reset_index()
)
tabla_unicos.columns.name = None

display(Markdown('### Conteo de valores únicos por fuente y columna'))
display(tabla_unicos)

### Conteo de valores únicos por fuente y columna

,fuente,canal_agendamiento,cita_id,cita_origen_id,confirmada,documento_identidad,edad,estado_crudo,fecha_actualizacion,fecha_cita,fecha_creacion,gestion_recuperacion,motivo_cierre,observaciones,paciente_id,recordatorio_enviado,regimen,sexo_crudo,telefono
0,NORTE,4,6357,357,2,0,98,6,5661,4818,5404,2,5,10,2255,2,3,2,0
1,OCCIDENTE,4,4228,228,2,0,98,6,3884,3509,3788,2,5,10,1513,2,3,2,0
2,SUR,4,5312,312,2,1885,98,6,4722,4177,4658,2,5,10,1885,2,3,2,1885


In [69]:
columnas_resumen = [
    'canal_agendamiento',
    'confirmada',
    'estado_crudo',
    'gestion_recuperacion',
    'motivo_cierre',
    'recordatorio_enviado',
    'regimen',
    'sexo_crudo',
]

resumen_dominios_sql = "\nUNION ALL\n".join([
    f"""
    SELECT
        fuente,
        '{columna}' AS columna,
        string_agg(DISTINCT CAST({columna} AS VARCHAR), ', ' ORDER BY CAST({columna} AS VARCHAR)) AS valores
    FROM
        citas_std
    WHERE
        {columna} IS NOT NULL
    GROUP BY
        fuente
    """.strip()
    for columna in columnas_resumen
])

resumen_dominios = conn.execute(resumen_dominios_sql).fetchdf()
tabla_dominios = (
    resumen_dominios
    .pivot(index='fuente', columns='columna', values='valores')
    .reset_index()
)
tabla_dominios.columns.name = None

display(Markdown('### Valores observados por fuente en columnas clave'))
display(tabla_dominios)

### Valores observados por fuente en columnas clave

,fuente,canal_agendamiento,confirmada,estado_crudo,gestion_recuperacion,motivo_cierre,recordatorio_enviado,regimen,sexo_crudo
0,NORTE,"call_center, voz, web, whatsapp","0, 1","ATENDIDA, CANCELADA, CONFIRMADA, NO_ASISTIO, PENDIENTE, REAGENDADA","0, 1","Cancelación por EPS, Médico no disponible, Paciente cancela por trabajo, Paciente cancela por viaje, Paciente no se ...","0, 1","Contributivo, Particular, Subsidiado","F, M"
1,OCCIDENTE,"call_center, voz, web, whatsapp","0, 1","ATD, CAN, CONF, NAS, PEN, REP","0, 1","Cancelación por EPS, Médico no disponible, Paciente cancela por trabajo, Paciente cancela por viaje, Paciente no se ...","0, 1","Contributivo, Particular, Subsidiado","Femenino, Masculino"
2,SUR,"call_center, voz, web, whatsapp","0, 1","ATENDIDA, CANCELADA, CONFIRMADA, NO_ASISTIO, PENDIENTE, REAGENDADA","0, 1","Cancelación por EPS, Médico no disponible, Paciente cancela por trabajo, Paciente cancela por viaje, Paciente no se ...","0, 1","Contributivo, Particular, Subsidiado","F, M"


In [70]:
estado_dist = conn.execute("""
    SELECT
        fuente, estado_crudo, COUNT(*) AS citas
    FROM
        citas_std
    GROUP BY
        1, 2
    ORDER BY
        fuente, citas DESC, estado_crudo
""").fetchdf()

sexo_dist = conn.execute("""
    SELECT
        fuente, sexo_crudo, COUNT(*) AS citas
    FROM
        citas_std
    GROUP BY
        1, 2
    ORDER BY
        fuente, citas DESC, sexo_crudo
""").fetchdf()

canal_dist = conn.execute("""
    SELECT
        fuente, canal_agendamiento, COUNT(*) AS citas
    FROM
        citas_std
    GROUP BY
        1, 2
    ORDER BY
        fuente, citas DESC, canal_agendamiento
""").fetchdf()

display(Markdown('### Distribución del estado crudo por fuente'))
display(estado_dist)
display(Markdown('### Distribución del sexo crudo por fuente'))
display(sexo_dist)
display(Markdown('### Distribución del canal de agendamiento por fuente'))
display(canal_dist)

### Distribución del estado crudo por fuente

,fuente,estado_crudo,citas
0,NORTE,ATENDIDA,4295
1,NORTE,NO_ASISTIO,748
2,NORTE,CANCELADA,744
3,NORTE,REAGENDADA,363
4,NORTE,PENDIENTE,225
5,NORTE,CONFIRMADA,180
6,OCCIDENTE,ATD,2772
7,OCCIDENTE,CAN,516
8,OCCIDENTE,NAS,478
9,OCCIDENTE,REP,228


### Distribución del sexo crudo por fuente

,fuente,sexo_crudo,citas
0,NORTE,F,3745
1,NORTE,M,2810
2,OCCIDENTE,Femenino,2388
3,OCCIDENTE,Masculino,1840
4,SUR,F,3016
5,SUR,M,2296


### Distribución del canal de agendamiento por fuente

,fuente,canal_agendamiento,citas
0,NORTE,voz,1987
1,NORTE,whatsapp,1657
2,NORTE,web,1586
3,NORTE,call_center,1325
4,OCCIDENTE,voz,1273
5,OCCIDENTE,whatsapp,1045
6,OCCIDENTE,web,1038
7,OCCIDENTE,call_center,872
8,SUR,voz,1549
9,SUR,whatsapp,1391


In [71]:
resumen_consistencia = pd.DataFrame([
    {'revision': 'mismo dominio de estado', 'resultado': 'No', 'detalle': 'Occidente usa ATD, CAN, CONF, NAS, PEN y REP, los demás usan un formato largo.'},
    {'revision': 'mismo dominio de sexo', 'resultado': 'No', 'detalle': 'Norte y Sur usan F/M. Occidente usa Femenino/Masculino.'},
    {'revision': 'mismo dominio de canal', 'resultado': 'Sí', 'detalle': 'Las tres IPS usan voz, whatsapp, web y call_center.'},
    {'revision': 'llave de cita comparable', 'resultado': 'Parcial', 'detalle': 'Se puede homologar, pero Occidente trae otro nombre de columna.'},
    {'revision': 'llave de paciente comparable entre IPS', 'resultado': 'No probado', 'detalle': 'No hubo coincidencias de paciente_id entre IPS; con esta extracción no se puede deduplicar entre clientes.'},
])

display(Markdown('### Consistencia entre fuentes'))
display(resumen_consistencia)

### Consistencia entre fuentes

,revision,resultado,detalle
0,mismo dominio de estado,No,"Occidente usa ATD, CAN, CONF, NAS, PEN y REP, los demás usan un formato largo."
1,mismo dominio de sexo,No,Norte y Sur usan F/M. Occidente usa Femenino/Masculino.
2,mismo dominio de canal,Sí,"Las tres IPS usan voz, whatsapp, web y call_center."
3,llave de cita comparable,Parcial,"Se puede homologar, pero Occidente trae otro nombre de columna."
4,llave de paciente comparable entre IPS,No probado,No hubo coincidencias de paciente_id entre IPS; con esta extracción no se puede deduplicar entre clientes.


In [72]:
expected_dict_fields = [
    'cita_id', 'paciente_id', 'edad', 'sexo', 'regimen', 'localidad', 'especialidad',
    'medico_id', 'sede', 'canal_agendamiento', 'fecha_creacion', 'fecha_cita',
    'fecha_actualizacion', 'estado', 'recordatorio_enviado', 'confirmada',
    'gestion_recuperacion', 'motivo_cierre', 'observaciones', 'cita_origen_id'
]

schema_sources = {
    'NORTE': conn.execute('DESCRIBE SELECT * FROM citas_norte_raw').fetchdf()['column_name'].tolist(),
    'SUR': conn.execute('DESCRIBE SELECT * FROM citas_sur_raw').fetchdf()['column_name'].tolist(),
    'OCCIDENTE': conn.execute('DESCRIBE SELECT * FROM citas_occidente_raw').fetchdf()['column_name'].tolist(),
}
schema_compare = pd.DataFrame([
    {'fuente': fuente, 'columnas_reales': sorted(columnas)}
    for fuente, columnas in schema_sources.items()
])
schema_compare['campos_no_esperados'] = schema_compare['columnas_reales'].apply(
    lambda cols: sorted([col for col in cols if col not in expected_dict_fields])
)
schema_compare['campos_esperados_ausentes'] = schema_compare['columnas_reales'].apply(
    lambda cols: sorted([col for col in expected_dict_fields if col not in cols])
)

display(Markdown('### Diferencias entre lo observado y el diccionario inicial'))
display(schema_compare[['fuente', 'campos_no_esperados', 'campos_esperados_ausentes']])

### Diferencias entre lo observado y el diccionario inicial

,fuente,campos_no_esperados,campos_esperados_ausentes
0,NORTE,[],[]
1,SUR,"[documento_identidad, telefono]",[]
2,OCCIDENTE,"[edad_paciente, estado_cita, fecha_hora_cita, genero, id_cita, id_cita_origen, id_paciente, regimen_salud]","[cita_id, cita_origen_id, edad, estado, fecha_cita, paciente_id, regimen, sexo]"


## 2. Hallazgos priorizados


In [73]:
total_citas = conn.execute('SELECT COUNT(*) FROM citas_std').fetchone()[0]
total_eventos = len(wa)
recordatorios_reportados = conn.execute(
    'SELECT COUNT(*) FROM citas_std WHERE recordatorio_enviado = 1'
).fetchone()[0]

csv_metricas = {
    'filas_occidente': conn.execute('SELECT COUNT(*) FROM citas_occidente').fetchone()[0],
    'filas_sur': conn.execute('SELECT COUNT(*) FROM citas_sur').fetchone()[0],
    'duplicados_norte': conn.execute("""
        SELECT
            COUNT(*)
        FROM
            citas_norte
        WHERE
            cita_id IN (
                SELECT cita_id
                FROM citas_norte
                GROUP BY 1
                HAVING COUNT(*) > 1
            )
    """).fetchone()[0],
    'fechas_inconsistentes': conn.execute("""
        SELECT
            COUNT(*)
        FROM
            citas_std
        WHERE
            fecha_cita < fecha_creacion OR fecha_actualizacion < fecha_creacion
    """).fetchone()[0],
    'fuera_corte': conn.execute(f"""
        SELECT
            COUNT(*)
        FROM
            citas_std
        WHERE
            fecha_creacion > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
            OR fecha_cita > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
            OR fecha_actualizacion > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
    """).fetchone()[0],
    'futuras_cerradas': conn.execute(f"""
        SELECT
            COUNT(*)
        FROM
            citas_std
        WHERE
            fecha_cita > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
            AND estado_homologado <> 'PENDIENTE'
    """).fetchone()[0],
}

cita_ids = set(
    conn.execute('SELECT DISTINCT cita_id FROM citas_std').fetchdf()['cita_id']
)
citas_con_recordatorio = conn.execute("""
    SELECT
        fuente,
        cita_id,
        estado_crudo AS estado,
        recordatorio_enviado,
        confirmada
    FROM
        citas_std
    WHERE
        recordatorio_enviado = 1
""").fetchdf()

mask_wa_sin_ref = wa['ref_cita'].isna()
mask_wa_ref_no_encontrada = wa['ref_cita'].notna() & ~wa['ref_cita'].isin(cita_ids)
mask_recordatorio_sin_wa = ~citas_con_recordatorio['cita_id'].isin(set(wa['ref_cita'].dropna()))

dup_wa = (
    wa.groupby(['tipo', 'ips', 'ref_cita', 'destinatario', 'ts'], dropna=False)
      .size()
      .reset_index(name='repeticiones')
)
dup_wa = dup_wa[dup_wa['repeticiones'] > 1]

hallazgos_resumen = pd.DataFrame([
    {
        'hallazgo': 'Estados no homologados en Occidente',
        'registros_afectados': csv_metricas['filas_occidente'],
        'porcentaje_sobre_base': round(csv_metricas['filas_occidente'] * 100 / total_citas, 2),
        'base': 'todas las citas',
        'impacto_en_negocio': 'Rompe la comparabilidad del ausentismo y del benchmark entre IPS.',
        'abordaje': 'Homologar catálogo de estado antes de calcular indicadores.'
    },
    {
        'hallazgo': 'Estructura distinta frente al diccionario y entre IPS',
        'registros_afectados': csv_metricas['filas_sur'] + csv_metricas['filas_occidente'],
        'porcentaje_sobre_base': round((csv_metricas['filas_sur'] + csv_metricas['filas_occidente']) * 100 / total_citas, 2),
        'base': 'todas las citas',
        'impacto_en_negocio': 'Aumenta el riesgo de leer mal columnas o de perder campos al unir fuentes.',
        'abordaje': 'Definir un esquema por fuente y una capa de homologación obligatoria.'
    },
    {
        'hallazgo': 'Dominio de sexo no homogéneo',
        'registros_afectados': csv_metricas['filas_occidente'],
        'porcentaje_sobre_base': round(csv_metricas['filas_occidente'] * 100 / total_citas, 2),
        'base': 'todas las citas',
        'impacto_en_negocio': 'Distorsiona cruces por perfil de paciente si no se normaliza antes.',
        'abordaje': 'Estandarizar a un catálogo único antes de analítica y modelado.'
    },
    {
        'hallazgo': 'Citas duplicadas o versionadas en Norte',
        'registros_afectados': csv_metricas['duplicados_norte'],
        'porcentaje_sobre_base': round(csv_metricas['duplicados_norte'] * 100 / conn.execute('SELECT COUNT(*) FROM citas_norte').fetchone()[0], 2),
        'base': 'filas de Norte',
        'impacto_en_negocio': 'Infla volumen y puede duplicar citas atendidas, incumplidas o confirmadas.',
        'abordaje': 'Definir si la extracción trae historia de estados y quedarse con la última versión por cita.'
    },
    {
        'hallazgo': 'Fechas imposibles en la secuencia de la cita',
        'registros_afectados': csv_metricas['fechas_inconsistentes'],
        'porcentaje_sobre_base': round(csv_metricas['fechas_inconsistentes'] * 100 / total_citas, 2),
        'base': 'todas las citas',
        'impacto_en_negocio': 'Daña tiempos de anticipación, oportunidad de agenda y modelos de predicción.',
        'abordaje': 'Bloquear registros donde cita o actualización queden antes de la creación.'
    },
    {
        'hallazgo': 'Fechas posteriores al corte declarado en el diccionario',
        'registros_afectados': csv_metricas['fuera_corte'],
        'porcentaje_sobre_base': round(csv_metricas['fuera_corte'] * 100 / total_citas, 2),
        'base': 'todas las citas',
        'impacto_en_negocio': 'Hace dudoso el periodo real de análisis y puede cambiar cualquier métrica histórica.',
        'abordaje': 'Acordar y documentar la fecha real de corte antes de publicar cifras.'
    },
    {
        'hallazgo': 'Citas futuras con estado distinto de pendiente',
        'registros_afectados': csv_metricas['futuras_cerradas'],
        'porcentaje_sobre_base': round(csv_metricas['futuras_cerradas'] * 100 / total_citas, 2),
        'base': 'todas las citas',
        'impacto_en_negocio': 'Puede mover el ausentismo o la atención a fechas donde la cita aún no debería cerrar.',
        'abordaje': 'Permitir estados cerrados solo cuando la cita ya pasó o cuando exista una regla formal para anticiparlos.'
    },
    {
        'hallazgo': 'Eventos de WhatsApp sin referencia de cita o con referencia que no existe',
        'registros_afectados': int((mask_wa_sin_ref | mask_wa_ref_no_encontrada).sum()),
        'porcentaje_sobre_base': round((mask_wa_sin_ref | mask_wa_ref_no_encontrada).sum() * 100 / total_eventos, 2),
        'base': 'todos los eventos de WhatsApp',
        'impacto_en_negocio': 'Impide medir contacto, entrega, lectura y confirmación por cita.',
        'abordaje': 'Hacer obligatoria la referencia de cita y validar que exista antes de guardar el evento.'
    },
    {
        'hallazgo': 'Citas marcadas con recordatorio enviado pero sin evento en WhatsApp',
        'registros_afectados': int(mask_recordatorio_sin_wa.sum()),
        'porcentaje_sobre_base': round(mask_recordatorio_sin_wa.sum() * 100 / recordatorios_reportados, 2),
        'base': 'citas con recordatorio_enviado = 1',
        'impacto_en_negocio': 'Sobreestima la cobertura del canal y puede ocultar fallas de mensajería.',
        'abordaje': 'Cruzar el indicador de envío solo contra eventos reales del canal.'
    },
    {
        'hallazgo': 'Eventos duplicados en WhatsApp',
        'registros_afectados': int(dup_wa['repeticiones'].sum()),
        'porcentaje_sobre_base': round(dup_wa['repeticiones'].sum() * 100 / total_eventos, 2),
        'base': 'todos los eventos de WhatsApp',
        'impacto_en_negocio': 'Infla métricas de entrega, lectura o respuesta y sesga el embudo de contacto.',
        'abordaje': 'Deduplicar por tipo, cita, destinatario y marca de tiempo, y revisar el origen de la repetición.'
    },
])

display(hallazgos_resumen)

,hallazgo,registros_afectados,porcentaje_sobre_base,base,impacto_en_negocio,abordaje
0,Estados no homologados en Occidente,4228,26.27,todas las citas,Rompe la comparabilidad del ausentismo y del benchmark entre IPS.,Homologar catálogo de estado antes de calcular indicadores.
1,Estructura distinta frente al diccionario y entre IPS,9540,59.27,todas las citas,Aumenta el riesgo de leer mal columnas o de perder campos al unir fuentes.,Definir un esquema por fuente y una capa de homologación obligatoria.
2,Dominio de sexo no homogéneo,4228,26.27,todas las citas,Distorsiona cruces por perfil de paciente si no se normaliza antes.,Estandarizar a un catálogo único antes de analítica y modelado.
3,Citas duplicadas o versionadas en Norte,395,6.03,filas de Norte,"Infla volumen y puede duplicar citas atendidas, incumplidas o confirmadas.",Definir si la extracción trae historia de estados y quedarse con la última versión por cita.
4,Fechas imposibles en la secuencia de la cita,122,0.76,todas las citas,"Daña tiempos de anticipación, oportunidad de agenda y modelos de predicción.",Bloquear registros donde cita o actualización queden antes de la creación.
5,Fechas posteriores al corte declarado en el diccionario,1202,7.47,todas las citas,Hace dudoso el periodo real de análisis y puede cambiar cualquier métrica histórica.,Acordar y documentar la fecha real de corte antes de publicar cifras.
6,Citas futuras con estado distinto de pendiente,620,3.85,todas las citas,Puede mover el ausentismo o la atención a fechas donde la cita aún no debería cerrar.,Permitir estados cerrados solo cuando la cita ya pasó o cuando exista una regla formal para anticiparlos.
7,Eventos de WhatsApp sin referencia de cita o con referencia que no existe,12069,31.42,todos los eventos de WhatsApp,"Impide medir contacto, entrega, lectura y confirmación por cita.",Hacer obligatoria la referencia de cita y validar que exista antes de guardar el evento.
8,Citas marcadas con recordatorio enviado pero sin evento en WhatsApp,410,3.63,citas con recordatorio_enviado = 1,Sobreestima la cobertura del canal y puede ocultar fallas de mensajería.,Cruzar el indicador de envío solo contra eventos reales del canal.
9,Eventos duplicados en WhatsApp,3590,9.35,todos los eventos de WhatsApp,"Infla métricas de entrega, lectura o respuesta y sesga el embudo de contacto.","Deduplicar por tipo, cita, destinatario y marca de tiempo, y revisar el origen de la repetición."


### Representación SQL de hallazgos de calidad de datos

In [74]:
duckdb_hallazgo_sql = {
    '01_estados_no_homologados.sql': f"""
        WITH citas_norte AS (
            SELECT 'NORTE' AS fuente, estado AS estado_crudo
            FROM read_csv_auto('{north_path}', header = true)
        ),
        citas_sur AS (
            SELECT 'SUR' AS fuente, estado AS estado_crudo
            FROM read_csv_auto('{sur_path}', header = true)
        ),
        citas_occidente AS (
            SELECT 'OCCIDENTE' AS fuente, estado_cita AS estado_crudo
            FROM read_csv_auto('{occ_path}', header = true, delim = ';')
        ),
        estados_unificados AS (
            SELECT * FROM citas_norte
            UNION ALL
            SELECT * FROM citas_sur
            UNION ALL
            SELECT * FROM citas_occidente
        )
        SELECT fuente, estado_crudo, COUNT(*) AS citas,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY fuente), 2) AS porcentaje_fuente
        FROM estados_unificados
        GROUP BY 1, 2
        ORDER BY fuente, citas DESC, estado_crudo
    """,
    '02_esquemas_y_campos_no_homogeneos.sql': f"""
        WITH norte AS (
            SELECT 'NORTE' AS fuente, column_name, column_type
            FROM (DESCRIBE SELECT * FROM read_csv_auto('{north_path}', header = true))
        ),
        sur AS (
            SELECT 'SUR' AS fuente, column_name, column_type
            FROM (DESCRIBE SELECT * FROM read_csv_auto('{sur_path}', header = true))
        ),
        occidente AS (
            SELECT 'OCCIDENTE' AS fuente, column_name, column_type
            FROM (DESCRIBE SELECT * FROM read_csv_auto('{occ_path}', header = true, delim = ';'))
        )
        SELECT * FROM norte
        UNION ALL
        SELECT * FROM sur
        UNION ALL
        SELECT * FROM occidente
        ORDER BY column_name, fuente
    """,
    '03_dominios_sexo_no_homologados.sql': f"""
        WITH norte AS (
            SELECT 'NORTE' AS fuente, sexo AS sexo_crudo
            FROM read_csv_auto('{north_path}', header = true)
        ),
        sur AS (
            SELECT 'SUR' AS fuente, sexo AS sexo_crudo
            FROM read_csv_auto('{sur_path}', header = true)
        ),
        occidente AS (
            SELECT 'OCCIDENTE' AS fuente, genero AS sexo_crudo
            FROM read_csv_auto('{occ_path}', header = true, delim = ';')
        ),
        sexo_unificado AS (
            SELECT * FROM norte
            UNION ALL
            SELECT * FROM sur
            UNION ALL
            SELECT * FROM occidente
        )
        SELECT fuente, sexo_crudo, COUNT(*) AS citas,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY fuente), 2) AS porcentaje_fuente
        FROM sexo_unificado
        GROUP BY 1, 2
        ORDER BY fuente, citas DESC, sexo_crudo
    """,
    '04_citas_duplicadas_norte.sql': f"""
        WITH norte AS (
            SELECT *
            FROM read_csv_auto('{north_path}', header = true)
        ),
        duplicados AS (
            SELECT
                cita_id,
                COUNT(*) AS filas_con_misma_cita,
                COUNT(DISTINCT hash(
                    paciente_id, edad, sexo, regimen, localidad, especialidad, medico_id, sede,
                    canal_agendamiento, fecha_creacion, fecha_cita, fecha_actualizacion, estado,
                    recordatorio_enviado, confirmada, gestion_recuperacion,
                    coalesce(motivo_cierre, ''), coalesce(observaciones, ''), coalesce(cita_origen_id, '')
                )) AS versiones_detectadas,
                MIN(fecha_actualizacion) AS primera_actualizacion,
                MAX(fecha_actualizacion) AS ultima_actualizacion
            FROM norte
            GROUP BY 1
            HAVING COUNT(*) > 1
        )
        SELECT cita_id, filas_con_misma_cita, versiones_detectadas,
               CASE WHEN versiones_detectadas = 1 THEN 'duplicado_exacto' ELSE 'versiones_conflictivas' END AS tipo_duplicado,
               primera_actualizacion, ultima_actualizacion
        FROM duplicados
        ORDER BY tipo_duplicado DESC, filas_con_misma_cita DESC, cita_id
    """,
    '05_fechas_inconsistentes.sql': f"""
        WITH norte AS (
            SELECT 'NORTE' AS fuente, cita_id, fecha_creacion, fecha_cita, fecha_actualizacion, estado
            FROM read_csv_auto('{north_path}', header = true)
        ),
        sur AS (
            SELECT 'SUR' AS fuente, cita_id, fecha_creacion, fecha_cita, fecha_actualizacion, estado
            FROM read_csv_auto('{sur_path}', header = true)
        ),
        occidente AS (
            SELECT 'OCCIDENTE' AS fuente, id_cita AS cita_id, fecha_creacion, fecha_hora_cita AS fecha_cita, fecha_actualizacion, estado_cita AS estado
            FROM read_csv_auto('{occ_path}', header = true, delim = ';')
        ),
        base AS (
            SELECT * FROM norte
            UNION ALL
            SELECT * FROM sur
            UNION ALL
            SELECT * FROM occidente
        )
        SELECT fuente, cita_id, fecha_creacion, fecha_cita, fecha_actualizacion, estado,
               CASE
                   WHEN CAST(fecha_cita AS TIMESTAMP) < CAST(fecha_creacion AS TIMESTAMP) THEN 'cita_antes_de_creacion'
                   WHEN CAST(fecha_actualizacion AS TIMESTAMP) < CAST(fecha_creacion AS TIMESTAMP) THEN 'actualizacion_antes_de_creacion'
               END AS inconsistencia
        FROM base
        WHERE CAST(fecha_cita AS TIMESTAMP) < CAST(fecha_creacion AS TIMESTAMP)
           OR CAST(fecha_actualizacion AS TIMESTAMP) < CAST(fecha_creacion AS TIMESTAMP)
        ORDER BY fuente, inconsistencia, fecha_creacion
    """,
    '06_corte_extraccion_inconsistente.sql': f"""
        WITH norte AS (
            SELECT 'NORTE' AS fuente, cita_id, fecha_creacion, fecha_cita, fecha_actualizacion
            FROM read_csv_auto('{north_path}', header = true)
        ),
        sur AS (
            SELECT 'SUR' AS fuente, cita_id, fecha_creacion, fecha_cita, fecha_actualizacion
            FROM read_csv_auto('{sur_path}', header = true)
        ),
        occidente AS (
            SELECT 'OCCIDENTE' AS fuente, id_cita AS cita_id, fecha_creacion, fecha_hora_cita AS fecha_cita, fecha_actualizacion
            FROM read_csv_auto('{occ_path}', header = true, delim = ';')
        ),
        base AS (
            SELECT * FROM norte
            UNION ALL
            SELECT * FROM sur
            UNION ALL
            SELECT * FROM occidente
        )
        SELECT fuente, cita_id, fecha_creacion, fecha_cita, fecha_actualizacion,
               CASE WHEN CAST(fecha_cita AS TIMESTAMP) > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}' THEN 1 ELSE 0 END AS cita_supera_corte,
               CASE WHEN CAST(fecha_creacion AS TIMESTAMP) > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}' THEN 1 ELSE 0 END AS creacion_supera_corte,
               CASE WHEN CAST(fecha_actualizacion AS TIMESTAMP) > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}' THEN 1 ELSE 0 END AS actualizacion_supera_corte
        FROM base
        WHERE CAST(fecha_creacion AS TIMESTAMP) > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
           OR CAST(fecha_cita AS TIMESTAMP) > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
           OR CAST(fecha_actualizacion AS TIMESTAMP) > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
        ORDER BY fuente, fecha_cita
    """,
    '07_citas_futuras_con_estado_cerrado.sql': f"""
        WITH norte AS (
            SELECT 'NORTE' AS fuente, cita_id, fecha_cita, estado
            FROM read_csv_auto('{north_path}', header = true)
        ),
        sur AS (
            SELECT 'SUR' AS fuente, cita_id, fecha_cita, estado
            FROM read_csv_auto('{sur_path}', header = true)
        ),
        occidente AS (
            SELECT 'OCCIDENTE' AS fuente, id_cita AS cita_id, fecha_hora_cita AS fecha_cita, estado_cita AS estado
            FROM read_csv_auto('{occ_path}', header = true, delim = ';')
        ),
        base AS (
            SELECT * FROM norte
            UNION ALL
            SELECT * FROM sur
            UNION ALL
            SELECT * FROM occidente
        )
        SELECT fuente, cita_id, fecha_cita, estado
        FROM base
        WHERE CAST(fecha_cita AS TIMESTAMP) > TIMESTAMP '{CORTE_EXTRACCION:%Y-%m-%d %H:%M:%S}'
          AND estado NOT IN ('PENDIENTE', 'PEN')
        ORDER BY fuente, fecha_cita, estado
    """,
}

wa_eventos_sin_cruce = wa.loc[
    mask_wa_sin_ref | mask_wa_ref_no_encontrada,
    ['_id', 'ips', 'tipo', 'ts', 'destinatario', 'ref_cita']
].copy()
wa_eventos_sin_cruce['problema_cruce'] = wa_eventos_sin_cruce['ref_cita'].apply(
    lambda value: 'sin_ref_cita' if pd.isna(value) else 'ref_cita_no_encontrada'
)
wa_eventos_sin_cruce = wa_eventos_sin_cruce.sort_values(['problema_cruce', 'ips', 'ts'])

citas_con_recordatorio_sin_evento = citas_con_recordatorio.loc[mask_recordatorio_sin_wa].sort_values(['fuente', 'cita_id'])
eventos_whatsapp_duplicados = dup_wa.sort_values(['repeticiones', 'tipo', 'ips', 'ref_cita', 'ts'], ascending=[False, True, True, True, True])

evidencias_pandas = {
    '08_whatsapp_eventos_sin_ref_o_sin_cita.sql': wa_eventos_sin_cruce,
    '09_citas_con_recordatorio_sin_evento_whatsapp.sql': citas_con_recordatorio_sin_evento,
    '10_eventos_whatsapp_duplicados.sql': eventos_whatsapp_duplicados,
}

def notebook_evidence(file_name: str) -> pd.DataFrame:
    if file_name in duckdb_hallazgo_sql:
        return conn.execute(duckdb_hallazgo_sql[file_name]).fetchdf()
    return evidencias_pandas[file_name]

hallazgos_detalle = [
    {'archivo': '01_estados_no_homologados.sql', 'titulo': 'Estados de cita distintos entre IPS', 'problema': 'Occidente usa abreviaturas y Norte/Sur usan el nombre largo del estado.', 'impacto': 'Sin homologación, el ausentismo y la atención no son comparables entre clientes.', 'abordaje': 'Crear una tabla de equivalencias y usarla antes de calcular indicadores.'},
    {'archivo': '02_esquemas_y_campos_no_homogeneos.sql', 'titulo': 'Esquemas y nombres de campo no homogéneos', 'problema': 'Sur trae campos extra y Occidente cambia nombres clave como id_cita y fecha_hora_cita.', 'impacto': 'Una lectura única de archivos puede fallar o mezclar campos equivocados.', 'abordaje': 'Definir contrato técnico por fuente y una homologación mínima antes de integrar.'},
    {'archivo': '03_dominios_sexo_no_homologados.sql', 'titulo': 'Dominio de sexo no alineado', 'problema': 'Occidente usa Femenino/Masculino y las otras IPS usan F/M.', 'impacto': 'Segmentaciones por sexo pueden quedar partidas en dos catálogos.', 'abordaje': 'Normalizar el campo a un catálogo único.'},
    {'archivo': '04_citas_duplicadas_norte.sql', 'titulo': 'Duplicados o versiones de la misma cita en Norte', 'problema': 'Norte repite cita_id y en varios casos cambia el estado o la fecha de actualización.', 'impacto': 'Puede inflar conteos y sesgar cualquier tasa operacional.', 'abordaje': 'Definir la lógica de última versión y deduplicar antes de analizar.'},
    {'archivo': '05_fechas_inconsistentes.sql', 'titulo': 'Secuencia de fechas imposible', 'problema': 'Hay casos donde la cita o la actualización queda antes de la creación.', 'impacto': 'Daña métricas de anticipación, puntualidad y preparación del modelo.', 'abordaje': 'Rechazar o marcar esos registros para revisión de origen.'},
    {'archivo': '06_corte_extraccion_inconsistente.sql', 'titulo': 'El periodo real no coincide con el corte declarado', 'problema': 'El diccionario habla de corte a 30 de junio de 2026, pero los datos llegan hasta agosto.', 'impacto': 'Cualquier tendencia mensual o comparación histórica puede salir corrida.', 'abordaje': 'Acordar el corte real antes de divulgar cifras.'},
    {'archivo': '07_citas_futuras_con_estado_cerrado.sql', 'titulo': 'Citas futuras con estados cerrados', 'problema': 'Hay citas posteriores al corte con estado distinto de pendiente.', 'impacto': 'Puede mover atención, cancelación o ausentismo a fechas que no corresponden.', 'abordaje': 'Restringir el uso de estados cerrados o aclarar la lógica de actualización anticipada.'},
    {'archivo': '08_whatsapp_eventos_sin_ref_o_sin_cita.sql', 'titulo': 'Eventos de WhatsApp que no se pueden amarrar a una cita', 'problema': 'Muchos eventos no traen ref_cita y otros la traen pero no existe en citas.', 'impacto': 'Se pierde trazabilidad del embudo de recordatorio a nivel de cita.', 'abordaje': 'Hacer obligatoria la referencia y validar su existencia al recibir el evento.'},
    {'archivo': '09_citas_con_recordatorio_sin_evento_whatsapp.sql', 'titulo': 'Marcador de recordatorio sin evidencia del evento', 'problema': 'Hay citas con recordatorio_enviado = 1 que no aparecen en WhatsApp.', 'impacto': 'La cobertura del canal puede quedar inflada.', 'abordaje': 'Cruzar el indicador con eventos reales o reconstruir el dato desde la bitácora del canal.'},
    {'archivo': '10_eventos_whatsapp_duplicados.sql', 'titulo': 'Eventos duplicados en el canal', 'problema': 'Existen eventos repetidos con la misma combinación de tipo, cita, destinatario y tiempo.', 'impacto': 'Infla pasos del embudo y vuelve poco confiable la medición del canal.', 'abordaje': 'Deduplicar en ingestión y revisar reintentos o reenvíos desde origen.'},
]

for item in hallazgos_detalle:
    detalle_md = '\n\n'.join([
        f"### {item['titulo']}",
        f"**Problema:** {item['problema']}",
        f"**Impacto en negocio:** {item['impacto']}",
        f"**Cómo abordarlo:** {item['abordaje']}",
        f"**SQL de respaldo:** `sql/diagnostico/{item['archivo']}`",
    ])
    display(Markdown(detalle_md))
    evidence = notebook_evidence(item['archivo'])
    print(f'Filas devueltas por la evidencia del notebook: {len(evidence):,}')
    display(evidence if len(evidence) <= 20 else evidence.head(10))

### Estados de cita distintos entre IPS

**Problema:** Occidente usa abreviaturas y Norte/Sur usan el nombre largo del estado.

**Impacto en negocio:** Sin homologación, el ausentismo y la atención no son comparables entre clientes.

**Cómo abordarlo:** Crear una tabla de equivalencias y usarla antes de calcular indicadores.

**SQL de respaldo:** `sql/diagnostico/01_estados_no_homologados.sql`

Filas devueltas por la evidencia del notebook: 18


,fuente,estado_crudo,citas,porcentaje_fuente
0,NORTE,ATENDIDA,4295,65.52
1,NORTE,NO_ASISTIO,748,11.41
2,NORTE,CANCELADA,744,11.35
3,NORTE,REAGENDADA,363,5.54
4,NORTE,PENDIENTE,225,3.43
5,NORTE,CONFIRMADA,180,2.75
6,OCCIDENTE,ATD,2772,65.56
7,OCCIDENTE,CAN,516,12.20
8,OCCIDENTE,NAS,478,11.31
9,OCCIDENTE,REP,228,5.39


### Esquemas y nombres de campo no homogéneos

**Problema:** Sur trae campos extra y Occidente cambia nombres clave como id_cita y fecha_hora_cita.

**Impacto en negocio:** Una lectura única de archivos puede fallar o mezclar campos equivocados.

**Cómo abordarlo:** Definir contrato técnico por fuente y una homologación mínima antes de integrar.

**SQL de respaldo:** `sql/diagnostico/02_esquemas_y_campos_no_homogeneos.sql`

Filas devueltas por la evidencia del notebook: 62


...

Filas devueltas por la evidencia del notebook: 410


,fuente,cita_id,estado,recordatorio_enviado,confirmada
1344,NORTE,NOR-182108,CANCELADA,1,1
394,NORTE,NOR-182188,ATENDIDA,1,0
3859,NORTE,NOR-182284,ATENDIDA,1,0
2564,NORTE,NOR-182293,REAGENDADA,1,0
52,NORTE,NOR-182332,ATENDIDA,1,0
2032,NORTE,NOR-182333,ATENDIDA,1,1
3099,NORTE,NOR-182334,ATENDIDA,1,0
4216,NORTE,NOR-182339,REAGENDADA,1,0
3020,NORTE,NOR-182381,CANCELADA,1,0
1407,NORTE,NOR-182565,NO_ASISTIO,1,0


### Eventos duplicados en el canal

**Problema:** Existen eventos repetidos con la misma combinación de tipo, cita, destinatario y tiempo.

**Impacto en negocio:** Infla pasos del embudo y vuelve poco confiable la medición del canal.

**Cómo abordarlo:** Deduplicar en ingestión y revisar reintentos o reenvíos desde origen.

**SQL de respaldo:** `sql/diagnostico/10_eventos_whatsapp_duplicados.sql`

Filas devueltas por la evidencia del notebook: 1,795


,tipo,ips,ref_cita,destinatario,ts,repeticiones
13,entregado,NORTE,NOR-182135,+573264607647,1753291200000,2
32,entregado,NORTE,NOR-182162,+573933452221,1755817740000,2
54,entregado,NORTE,NOR-182214,+573597520967,1758705780000,2
58,entregado,NORTE,NOR-182219,+573313612263,1755033720000,2
85,entregado,NORTE,NOR-182261,+573393429701,1784262600000,2
93,entregado,NORTE,NOR-182277,+573833724848,1768382700000,2
99,entregado,NORTE,NOR-182292,+573411102904,1784431500000,2
112,entregado,NORTE,NOR-182321,+573154632140,1755481440000,2
196,entregado,NORTE,NOR-182511,+573982756302,1752872880000,2
237,entregado,NORTE,NOR-182598,+573518925155,1780199520000,2


## 3. Reglas de calidad sugeridas

In [75]:
reglas_calidad = pd.DataFrame([
    {
        'problema': 'Estados crudos distintos entre IPS',
        'validacion': 'El estado crudo debe existir en un catálogo permitido por fuente y debe homologarse a un catálogo común.',
        'umbral': '0 filas fuera de catálogo',
        'accion_si_falla': 'Bloquear carga a capa analítica hasta homologar.'
    },
    {
        'problema': 'Esquema no esperado',
        'validacion': 'Cada fuente debe cumplir su esquema de columnas, nombres y tipos.',
        'umbral': '0 columnas faltantes en llaves y fechas',
        'accion_si_falla': 'Enviar alerta y bloquear carga a capa analítica hasta homologar.'
    },
    {
        'problema': 'Sexo en dominios distintos',
        'validacion': 'El sexo homologado debe quedar en el catálogo acordado.',
        'umbral': '100% de filas homologables',
        'accion_si_falla': 'Marcar filas y revisar mapeo.'
    },
    {
        'problema': 'Duplicados o versiones sin control',
        'validacion': 'Cada cita_id debe tener una sola versión vigente en la capa lista para análisis.',
        'umbral': '0 cita_id duplicados en la capa analítica',
        'accion_si_falla': 'Conservar solo la última versión y reportar origen.'
    },
    {
        'problema': 'Secuencia de fechas imposible',
        'validacion': 'fecha_creacion <= fecha_actualizacion y fecha_creacion <= fecha_cita.',
        'umbral': '0 filas inválidas',
        'accion_si_falla': 'Aislar registros y pedir corrección al origen.'
    },
    {
        'problema': 'Datos fuera del corte esperado',
        'validacion': 'Ninguna fecha debe superar el corte declarado para el lote.',
        'umbral': '0 filas fuera de corte',
        'accion_si_falla': 'Detener publicación hasta aclarar el periodo real.'
    },
    {
        'problema': 'Citas futuras con estado cerrado',
        'validacion': 'Si fecha_cita > fecha_de_corte, el estado debe ser pendiente salvo regla aprobada.',
        'umbral': '0 filas incompatibles',
        'accion_si_falla': 'Marcar inconsistencias y excluir del cálculo oficial.'
    },
    {
        'problema': 'Eventos sin referencia de cita',
        'validacion': 'Todo evento de WhatsApp debe traer ref_cita y esta debe existir en citas.',
        'umbral': '0 eventos sin llave válida',
        'accion_si_falla': 'No usar esos eventos en métricas de contacto.'
    },
    {
        'problema': 'Indicador de recordatorio sin soporte del canal',
        'validacion': 'Si recordatorio_enviado = 1, debe existir al menos un evento del canal para esa cita.',
        'umbral': '0 citas huérfanas',
        'accion_si_falla': 'Recalcular el indicador desde eventos reales.'
    },
    {
        'problema': 'Eventos duplicados',
        'validacion': 'No debe repetirse la misma combinación de tipo, cita, destinatario y marca de tiempo.',
        'umbral': '0 duplicados',
        'accion_si_falla': 'Deduplicar e investigar con el origen de datos.'
    },
])

display(reglas_calidad)

,problema,validacion,umbral,accion_si_falla
0,Estados crudos distintos entre IPS,El estado crudo debe existir en un catálogo permitido por fuente y debe homologarse a un catálogo común.,0 filas fuera de catálogo,Bloquear carga a capa analítica hasta homologar.
1,Esquema no esperado,"Cada fuente debe cumplir su esquema de columnas, nombres y tipos.",0 columnas faltantes en llaves y fechas,Enviar alerta y bloquear carga a capa analítica hasta homologar.
2,Sexo en dominios distintos,El sexo homologado debe quedar en el catálogo acordado.,100% de filas homologables,Marcar filas y revisar mapeo.
3,Duplicados o versiones sin control,Cada cita_id debe tener una sola versión vigente en la capa lista para análisis.,0 cita_id duplicados en la capa analítica,Conservar solo la última versión y reportar origen.
4,Secuencia de fechas imposible,fecha_creacion <= fecha_actualizacion y fecha_creacion <= fecha_cita.,0 filas inválidas,Aislar registros y pedir corrección al origen.
5,Datos fuera del corte esperado,Ninguna fecha debe superar el corte declarado para el lote.,0 filas fuera de corte,Detener publicación hasta aclarar el periodo real.
6,Citas futuras con estado cerrado,"Si fecha_cita > fecha_de_corte, el estado debe ser pendiente salvo regla aprobada.",0 filas incompatibles,Marcar inconsistencias y excluir del cálculo oficial.
7,Eventos sin referencia de cita,Todo evento de WhatsApp debe traer ref_cita y esta debe existir en citas.,0 eventos sin llave válida,No usar esos eventos en métricas de contacto.
8,Indicador de recordatorio sin soporte del canal,"Si recordatorio_enviado = 1, debe existir al menos un evento del canal para esa cita.",0 citas huérfanas,Recalcular el indicador desde eventos reales.
9,Eventos duplicados,"No debe repetirse la misma combinación de tipo, cita, destinatario y marca de tiempo.",0 duplicados,Deduplicar e investigar con el origen de datos.


## 4. Preguntas para el dueño de negocio

1. ¿La extracción de Norte trae el estado actual de cada cita o trae historia de cambios por la misma cita?
2. ¿Qué definición oficial de ausentismo se va a usar: solo `NO_ASISTIO`, o también hay reglas para cancelaciones tardías y reprogramaciones?
3. ¿`recordatorio_enviado` debe representar un intento de envío o un evento real confirmado por el canal?
4. ¿Es obligatorio que todos los eventos de WhatsApp tengan `ref_cita`, o existen mensajes del canal que no nacen de una cita?
5. ¿`paciente_id` está seudonimizado de la misma forma entre IPS o solo sirve dentro de cada cliente?
6. ¿Documento y teléfono deben existir solo en Sur o deberían estar disponibles en todas las IPS cuando el dato exista?